# e-TSL — แหล่งข้อมูลคำและท่าทางเลือก
ดึง metadata จริงจาก endpoint ที่หน้าเว็บใช้งาน เก็บนอก bank เพื่อ audit ก่อนเพิ่ม ไม่ใช่ผล recognition

In [ ]:
from pathlib import Path
import sys, json
from IPython.display import display
import pandas as pd
start=Path.cwd().resolve()
HERE=next((q for p in [start,*start.parents] for q in [p,p/"Boom/data_sources/e_tsl"] if (q/"scrape.py").is_file()),None)
if HERE is None: raise FileNotFoundError("เปิดจาก repo หรือโฟลเดอร์ e_tsl")
sys.path.insert(0,str(HERE))
import scrape
summary=scrape.build(refresh=False)
display(pd.DataFrame([summary]))

**อ่านผล:** 2,049 รายการไม่ใช่ 2,049 คำใหม่ทั้งหมด มีชื่อคำตรงกับ concepts เดิม 1,394 รายการ การนับชื่อคำไม่ยืนยันว่าคลิปซ้ำหรือผู้ทำท่าเป็นคนใหม่ ต้องเทียบ source IDs/hash เพิ่ม

In [ ]:
words=pd.read_csv(HERE/"words.csv")
variants=pd.read_csv(HERE/"sign_variants.csv")
display(words.head(20))
display(pd.read_csv(HERE/"priority_words.csv"))

**อ่านผล:** exact คือชื่อคำตรงกัน ส่วน substring_review_required เป็นเพียงข้อความมีคำที่ค้นอยู่ ห้ามรวมเป็น concept เดียวโดยอัตโนมัติ เช่น คำประสมหรือความหมายต่างกัน

In [ ]:
checks=HERE/"priority_url_checks.csv"
if checks.exists(): display(pd.read_csv(checks))
# ตรวจซ้ำเฉพาะเมื่อจำเป็น: scrape.check_priority()
# รีเฟรช metadata ทั้งเว็บหนึ่ง request: scrape.build(refresh=True)

## ใช้กับงานของเรา
เริ่มจาก ไป/ภาษามือ/กิน/คุณ และคู่สับสนที่พบใน tune เปิดวิดีโอตรวจท่าและความหมาย จับคู่ concept แบบตรวจรับรอง เปรียบเทียบกับ bank เดิม แล้วจึงสกัด landmarks/embedding เป็น experiment bank แยกไฟล์ ห้ามใช้ test เลือกคลิปเพิ่ม

รหัส root และรูปสัญลักษณ์มีประโยชน์ต่อการบรรยายท่า แต่ยังไม่ได้แปลเป็นกำ/แบ/ทิศฝ่ามือหรือ 53 attributes ต้องตรวจความหมายก่อน ไม่มีการดาวน์โหลดวิดีโอทั้งคลังหรือเปลี่ยน production ในขั้นนี้

## ดาวน์โหลดคำเป้าหมายและตรวจวิดีโอซ้ำ
ใช้ prepare_priority.py กับ Python environment ของโปรเจกต์ เนื่องจาก cv2 ใน kernel อื่นอาจไม่เข้ากับ NumPy อ่านผลที่มีแล้วก่อน ไม่ต้องดาวน์โหลดซ้ำ

In [ ]:
import subprocess
PY=scrape.ROOT/"Boom/venv-sign2say/bin/python"
RUN_DOWNLOAD=False
if RUN_DOWNLOAD:
    subprocess.run([str(PY),"-B",str(HERE/"prepare_priority.py")],check=True,cwd=scrape.ROOT)
file=HERE/"priority_download_audit.csv"
if file.exists(): display(pd.read_csv(file))
summary_path=HERE/"download_summary.json"
if summary_path.exists(): display(pd.DataFrame([json.loads(summary_path.read_text())]))

**อ่านผล:** exact duplicate ยืนยันจาก SHA256 เท่านั้น ชื่อคำตรงกันไม่เท่ากับคลิปซ้ำ และไม่พบ hash ซ้ำไม่ได้พิสูจน์ว่าเป็นคลิปใหม่เมื่อวิดีโอเก่าขาดหรือถูก encode ใหม่ เปิด local_review.html เพื่อดูคลิปที่ดาวน์โหลดจริง ยังไม่รวมเข้า bank

## ใช้ชื่อจากเว็บเป็น label
label และ source_label เก็บ wrd_word_th ตรงต้นทาง ไม่ตัดชื่อ ไม่แทนด้วย target ซึ่งเป็นเพียงคำค้น mapped_concept_id แยกต่างหาก และยังเป็น null จนตรวจรับรอง การจับคู่ชื่ออัตโนมัติใน words.csv เป็นเพียง candidate mapping ไม่ใช่ label

In [ ]:
import source_labels
display(pd.DataFrame([source_labels.build()]))
display(pd.read_csv(HERE/"clip_manifest.csv")[["clip_id","label","source_label","mapped_concept_id","mapping_status","path"]])

## เพิ่ม 12 คลิปเข้า experiment bank
ใช้ add_experiment_bank.py: MediaPipe 25 fps → Standard Schema → canonical hands → prep → encoder ONNX เดิม → L2 normalize แล้ว append ทุกแถว ใช้ label จากเว็บตรงกับ concept เดิม ไม่เทรนใหม่ ไม่แก้ production หรือ eval bank และไม่เพิ่มคลิปเข้าชุด test

ใช้ทั้งคลิปคำเดี่ยวเป็น span จึงยังอาจมีช่วงพักมือ ห้ามอ้างว่าขอบเขตผ่านการตรวจคนแล้ว ผู้ทำท่ายังไม่ยืนยัน identity จึงระบุ unknown:etsl ไม่ตีความว่าเพิ่ม signer ใหม่

In [ ]:
bank_dir=HERE/"experiment_bank"
report_path=bank_dir/"BANK_REPORT.json"
if report_path.exists():
    bank_report=json.loads(report_path.read_text())
    display(pd.DataFrame([{k:v for k,v in bank_report.items() if k not in ["clips","hashes"]}]))
    display(pd.DataFrame(bank_report["clips"]))
else:
    print("ยังไม่เสร็จ: ดู bank_add.log")

## ทดลองใช้ bank ใหม่แบบแยก
โค้ดด้านล่างปิดไว้เป็นค่าเริ่มต้น ต้องรันด้วย environment ของโปรเจกต์ ใช้โมเดล/config เดิมและเปลี่ยนเฉพาะ bank_file ไม่แก้เว็บอัตโนมัติ ประเมินบน dev/tune ชุดที่กำหนดล่วงหน้า ไม่ย้อนจูนจาก test ที่เปิดแล้ว

In [ ]:
LOAD_EXPERIMENT_BANK=False
if LOAD_EXPERIMENT_BANK:
    from modules.pipeline import SignPipeline
    pipe=SignPipeline(backend="onnx",device="cpu",vocab="conversation",use_llm=False,use_face=False,bank_file=bank_dir/"bank_etsl12.npz")